In [1]:
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pandas as pd
import math
from copy import deepcopy

In [2]:

# Your current debts
DEBTS = [
    # Credit Cards (revolving debt - highest rates)
    {
        "name": "SBI Credit Card",
        "type": "credit_card",
        "balance": 50000,
        "apr": 42,
        "min_payment": 2500,  # Fixed minimum or % will be calculated
    },
    
    # Loans (installment debt - fixed tenure)
    {
        "name": "HDFC Home Loan",
        "type": "loan",
        "balance": 2500000,
        "apr": 8.5,
        "tenure_months": 240,
        "emi": 21797  # Pre-calculated EMI
    },
    {
        "name": "Axis Car Loan",
        "type": "loan",
        "balance": 600000,
        "apr": 9.5,
        "tenure_months": 60,
        "emi": 12618
    },
    {
        "name": "Education Loan",
        "type": "loan",
        "balance": 400000,
        "apr": 10.5,
        "tenure_months": 84,
        "emi": 6890,
        "moratorium": False  # Set True if still in grace period
    },
]


In [3]:
# Your repayment capacity
MONTHLY_BUDGET = 60000  # Total you can afford per month
EXTRA_MONTHLY = 10000   # Extra beyond minimums (adjust this!)


# One-time windfalls
LUMP_SUM_PAYMENTS = [
    {"month": 1, "amount": 5000, "strategy": "target"},   # Diwali bonus
    {"month": 2, "amount": 20000, "strategy": "target"},  # Year-end bonus
]

In [5]:

def calculate_emi(principal, apr, months):
    """Standard EMI formula"""
    if months == 0 or principal == 0:
        return 0
    r = apr / 1200
    if r == 0:
        return principal / months
    return principal * r * math.pow(1 + r, months) / (math.pow(1 + r, months) - 1)

def simulate_payoff(debts, strategy, extra_monthly, lump_sums):
    """
    Simulate debt payoff with given strategy
    Returns: months_to_freedom, total_interest, timeline
    """
    debts = deepcopy(debts)  # Don't modify original
    month = 0
    total_interest = 0
    timeline = []
    
    # Calculate minimum monthly obligations
    min_required = sum(
        d.get("emi", d.get("min_payment", d["balance"] * 0.05)) 
        for d in debts if d["balance"] > 0
    )
    
    while any(d["balance"] > 0 for d in debts):
        month += 1
        if month > 600:  # Safety: 50 years max
            break
            
        month_interest = 0
        month_principal = 0
        
        # 1. Calculate interest for all debts
        for debt in debts:
            if debt["balance"] <= 0:
                continue
                
            monthly_rate = debt["apr"] / 1200
            interest = debt["balance"] * monthly_rate
            debt["balance"] += interest
            month_interest += interest
        
        # 2. Make minimum payments
        for debt in debts:
            if debt["balance"] <= 0:
                continue
                
            if debt["type"] == "credit_card":
                min_pay = max(debt.get("min_payment", 0), debt["balance"] * 0.05)
            else:
                min_pay = debt.get("emi", 0)
            
            payment = min(min_pay, debt["balance"])
            debt["balance"] -= payment
            month_principal += payment
        
        # 3. Apply lump sum if scheduled
        lump_sum_this_month = sum(
            ls["amount"] for ls in lump_sums 
            if ls["month"] == month
        )
        
        # 4. Distribute extra payment based on strategy
        extra_available = extra_monthly + lump_sum_this_month
        
        while extra_available > 0:
            # Find target debt based on strategy
            active_debts = [d for d in debts if d["balance"] > 0]
            if not active_debts:
                break
            
            if strategy == "avalanche":
                target = max(active_debts, key=lambda d: d["apr"])
            elif strategy == "snowball":
                target = min(active_debts, key=lambda d: d["balance"])
            elif strategy == "hybrid":
                # Attack high-interest first (>15%), then snowball rest
                high_interest = [d for d in active_debts if d["apr"] > 15]
                if high_interest:
                    target = max(high_interest, key=lambda d: d["apr"])
                else:
                    target = min(active_debts, key=lambda d: d["balance"])
            else:  # custom - defined by user
                target = active_debts[0]
            
            payment = min(extra_available, target["balance"])
            target["balance"] -= payment
            month_principal += payment
            extra_available -= payment
        
        total_interest += month_interest
        
        # Record snapshot
        timeline.append({
            "month": month,
            "total_balance": sum(d["balance"] for d in debts),
            "interest_paid": month_interest,
            "principal_paid": month_principal
        })
    
    return month, total_interest, timeline, debts


def format_duration(months):
    """Convert months to years/months"""
    years = months // 12
    remaining_months = months % 12
    if years == 0:
        return f"{remaining_months} months"
    elif remaining_months == 0:
        return f"{years} years"
    else:
        return f"{years} years, {remaining_months} months"

def get_debt_free_date(months_from_now):
    """Calculate future date"""
    future = datetime.now() + relativedelta(months=months_from_now)
    return future.strftime("%B %Y")


In [6]:

# ================= RUN COMPARISONS =================

print("=" * 80)
print("🎯 DEBT PAYOFF STRATEGY COMPARISON")
print("=" * 80)
print(f"\nYour Situation:")
print(f"  Total Debt:        ₹{sum(d['balance'] for d in DEBTS):,.0f}")
print(f"  Monthly Budget:    ₹{MONTHLY_BUDGET:,.0f}")
print(f"  Extra Monthly:     ₹{EXTRA_MONTHLY:,.0f}")
print(f"  Planned Windfalls: ₹{sum(ls['amount'] for ls in LUMP_SUM_PAYMENTS):,.0f}")

strategies = {
    "Avalanche (Highest Interest First)": "avalanche",
    "Snowball (Smallest Balance First)": "snowball",
    "Hybrid (High APR, then Snowball)": "hybrid",
}

results = {}

for name, strategy in strategies.items():
    months, interest, timeline, final_debts = simulate_payoff(
        DEBTS, strategy, EXTRA_MONTHLY, LUMP_SUM_PAYMENTS
    )
    results[name] = {
        "months": months,
        "interest": interest,
        "timeline": timeline
    }

# Find best strategy
best_strategy = min(results.items(), key=lambda x: x[1]["interest"])
baseline = results["Avalanche (Highest Interest First)"]

print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80)

comparison_data = []
for name, result in results.items():
    savings_vs_baseline = baseline["interest"] - result["interest"]
    time_diff = baseline["months"] - result["months"]
    
    comparison_data.append({
        "Strategy": name.split(" (")[0],
        "Payoff Time": format_duration(result["months"]),
        "Debt-Free Date": get_debt_free_date(result["months"]),
        "Total Interest": f"₹{result['interest']:,.0f}",
        "vs Avalanche": f"₹{savings_vs_baseline:+,.0f}" if name != "Avalanche (Highest Interest First)" else "Baseline",
        "Time Saved": f"{time_diff:+d} months" if name != "Avalanche (Highest Interest First)" else "-"
    })

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))

print("\n" + "=" * 80)
print(f"🏆 RECOMMENDED: {best_strategy[0]}")
print("=" * 80)
print(f"  Debt-Free Date:    {get_debt_free_date(best_strategy[1]['months'])}")
print(f"  Total Payoff Time: {format_duration(best_strategy[1]['months'])}")
print(f"  Total Interest:    ₹{best_strategy[1]['interest']:,.0f}")
print(f"  Monthly Payment:   ₹{MONTHLY_BUDGET:,.0f} (₹{EXTRA_MONTHLY:,.0f} extra)")

# ================= ATTACK ORDER =================

print("\n" + "=" * 80)
print("🎯 ATTACK ORDER (Avalanche Method)")
print("=" * 80)

sorted_debts = sorted(DEBTS, key=lambda d: d["apr"], reverse=True)
for i, debt in enumerate(sorted_debts, 1):
    debt_type = "💳" if debt["type"] == "credit_card" else "🏦"
    print(f"{i}. {debt_type} {debt['name']:25} | APR: {debt['apr']:5.1f}% | Balance: ₹{debt['balance']:>10,.0f}")

print("\n" + "=" * 80)
print("💡 OPTIMIZATION TIPS")
print("=" * 80)

# Calculate impact of different extra payment amounts
test_extras = [5000, 10000, 15000, 20000]
print("\nImpact of Extra Monthly Payments:")
for extra in test_extras:
    months, interest, _, _ = simulate_payoff(DEBTS, "avalanche", extra, LUMP_SUM_PAYMENTS)
    savings = baseline["interest"] - interest
    time_saved = baseline["months"] - months
    print(f"  +₹{extra:>6,}/month → Save ₹{savings:>8,.0f} | Free {time_saved:>2d} months earlier")

# Show debt composition
print("\n" + "=" * 80)
print("📈 DEBT BREAKDOWN")
print("=" * 80)
total_debt = sum(d["balance"] for d in DEBTS)
cc_debt = sum(d["balance"] for d in DEBTS if d["type"] == "credit_card")
loan_debt = sum(d["balance"] for d in DEBTS if d["type"] == "loan")

print(f"Credit Cards (High Interest): ₹{cc_debt:>12,.0f} ({cc_debt/total_debt*100:.1f}%)")
print(f"Loans (Fixed Term):           ₹{loan_debt:>12,.0f} ({loan_debt/total_debt*100:.1f}%)")
print(f"Total Debt:                   ₹{total_debt:>12,.0f}")

print("\n" + "=" * 80)
print("🔧 TO CUSTOMIZE:")
print("=" * 80)
print("1. Edit DEBTS list - add/remove/update your debts")
print("2. Set MONTHLY_BUDGET - total you can afford")
print("3. Set EXTRA_MONTHLY - amount beyond minimums")
print("4. Add LUMP_SUM_PAYMENTS - bonuses, tax refunds, etc.")
print("5. Run and compare strategies!")
print("=" * 80)

🎯 DEBT PAYOFF STRATEGY COMPARISON

Your Situation:
  Total Debt:        ₹3,550,000
  Monthly Budget:    ₹60,000
  Extra Monthly:     ₹10,000
  Planned Windfalls: ₹25,000

📊 STRATEGY COMPARISON
 Strategy        Payoff Time Debt-Free Date Total Interest vs Avalanche Time Saved
Avalanche 12 years, 2 months   January 2038     ₹1,870,895     Baseline          -
 Snowball 12 years, 2 months   January 2038     ₹1,870,895          ₹+0  +0 months
   Hybrid 12 years, 2 months   January 2038     ₹1,870,895          ₹+0  +0 months

🏆 RECOMMENDED: Avalanche (Highest Interest First)
  Debt-Free Date:    January 2038
  Total Payoff Time: 12 years, 2 months
  Total Interest:    ₹1,870,895
  Monthly Payment:   ₹60,000 (₹10,000 extra)

🎯 ATTACK ORDER (Avalanche Method)
1. 💳 SBI Credit Card           | APR:  42.0% | Balance: ₹    50,000
2. 🏦 Education Loan            | APR:  10.5% | Balance: ₹   400,000
3. 🏦 Axis Car Loan             | APR:   9.5% | Balance: ₹   600,000
4. 🏦 HDFC Home Loan            | A